# Evaluation of Formulas from First Order Logic 

In this notebook, we show how formulas from *first order logic* can be evaluated in Java.

## The Axioms of Group Theory

To have a nontrivial example of formulas, we use the formulas from [group theory](https://en.wikipedia.org/wiki/Group_theory).
A group is defined as a triple $ \langle G, \mathrm{e}, \circ \rangle $
where:
- $G$ is a non-empty set,
- $\mathrm{e}$ is an element from $G$, and
- $\circ:G \times G \rightarrow G$ is a binary function on $G$.

Furthermore, the following axioms have to be satisfied:
  * $\forall x: \mathrm{e} \circ x = x$
  * $\forall x: \exists y: y \circ x = \mathrm{e}$
  * $\forall x: \forall y: \forall z: (x \circ y) \circ z = x \circ (y \circ z)$

A group is *commutative* if, additionally, the following formula is satisfied:
  * $\forall x: \forall y: x \circ y = y \circ x$

We use the class `RecursiveSet` and the parser for first-order logic that is defined in the file `FOL-Parser.jsh`.  This parser represents terms and formulas as trees of *records*, which are discussed in the notebook `FOL-Parser.ipynb`:
* A term is either a variable `Var(x)` or a function application `App(f, args)`, where `args` is a list of terms.
* An atomic formula $p(t_1, \cdots, t_n)$ is represented as `Atom(p, args)`.
* The other formulas are represented as `Verum()`, `Falsum()`, `Not(f)`, `And(f, g)`, `Or(f, g)`, `Implies(f, g)`, `Equiv(f, g)`, `ForAll(x, f)`, and `Exists(x, f)`.

In [ ]:
%load ../lib/RecursiveSet.jsh
%load FOL-Parser.jsh

In [ ]:
@SafeVarargs
<T> RecursiveSet<T> set(T... elements) {
    return new RecursiveSet<>(elements);
}

Our new parser distinguishes between variables and function/predicate symbols as follows:
- *Variables* start with an **upper case** letter (e.g., `X`, `Y`).
- *Functions* and *predicates* start with a **lower case** letter (e.g., `e`, `p`).
- We can use the arithmetic operators `+`, `-`, `*`, `/`, `%`, `**` as infix operators.
- We can use the relational operators `<`, `>`, `=`, `≤`, `≥` as infix operators.
- The logical symbols `⊤`, `⊥`, `∧`, `∨`, `¬`, `→`, `↔`, `∀`, `∃` are supported with the usual precedences.

  The quantifiers `∀`, `∃` bind stronger than `∧` and `∨`.

  **Note** that `∧` binds stronger than `∨`.  However, in order to avoid confusion 
  we will not mix `∧` and `∨` in this lesson. 

We represent the symbols from group theory as follows:
- The neutral element "$\mathrm{e}$" is the 0-ary function `e`.
- The operator "$\circ$" is represented by the arithmetic operator `*`.
- Equality "$=$" is natively supported by the parser as `=`.

Then the formulas of group theory can be written as shown below.

In [ ]:
var G1 = "∀X: e * X = X";
var G2 = "∀X: ∃Y: Y * X = e";
var G3 = "∀X: ∀Y: ∀Z: (X * Y) * Z = X * (Y * Z)";
var G4 = "∀X: ∀Y: X * Y = Y * X";

The function `structure`, which is defined in `FOL-Parser.jsh`, shows the records a formula is built from.

In [ ]:
var F1 = parse(G1);
System.out.println(F1);
System.out.println(structure(F1));

In [ ]:
var F2 = parse(G2);
System.out.println(F2);
System.out.println(structure(F2));

In [ ]:
var F3 = parse(G3);
System.out.println(F3);
System.out.println(structure(F3));

In [ ]:
var F4 = parse(G4);
System.out.println(F4);
System.out.println(structure(F4));

## A Structure for Group Theory

A *tuple* of values is represented as a `List<Integer>`.  As lists are compared by value, tuples can be used as keys of maps and as elements of sets.  We use maps to map tuples of arguments to their function results and sets of tuples to represent the interpretation of predicates.
To simplify matters, we assume that the universe is a set of numbers.

Assume a signature 
$$ \Sigma = \langle \mathcal{V}, \mathcal{F}, \mathcal{P}, \texttt{arity} \rangle $$
is given. A *$\Sigma$-structure* $\mathcal{S}$ is a
pair $\langle \mathcal{U}, \mathcal{J} \rangle$, such that the following holds:
 * $\mathcal{U}$ is a non-empty set. This set is also called the
   *universe* of the $\Sigma$-structure. This universe contains the *values*.
 * $\mathcal{J}$ is the *interpretation* of the function and predicate symbols.

   Formally, we define $\mathcal{J}$ as a mapping with the following properties:
   + Every function symbol $c \in \mathcal{F}$ with $\texttt{arity}(c) = 0$ is mapped to an element
     of $\mathcal{U}$, i.e.we have $\mathcal{J}(c) \in \mathcal{U}$.  
     
     Instead of writing $\mathcal{J}(c)$ we will write $c^{\mathcal{J}}$.
   + Every function symbol $f \in \mathcal{F}$ with $\texttt{arity}(f) = m$ and $m > 0$ is mapped to
     an $m$-ary function
     $$ f^\mathcal{J}\colon \mathcal{U}^m \rightarrow \mathcal{U}$$
     that maps $m$-tuples of the universe $\mathcal{U}$ into the universe $\mathcal{U}$.
   + Every predicate symbol $p \in \mathcal{P}$ with $\texttt{arity}(p) = 0$ is mapped to
     the set $\mathbb{B}$ of truth values, i.e. $p^{\mathcal{J}} \in \{ \mathtt{true}, \mathtt{false}\}$.
   + Every predicate symbol $p \in \mathcal{P}$ with $\texttt{arity}(p) = n$ such that $n > 0$ is mapped to
     a subset 
     $$ p^\mathcal{J} \subseteq \mathcal{U}^n. $$

     The idea is that an atomic formula of the form $p(t_1, \cdots, t_n)$
     is interpreted as $\texttt{true}$ exactly if and only if the interpretation of the tuple
     $\langle t_1, \cdots, t_n \rangle$ is an element of the set $p^\mathcal{J}$.
   + If the symbol `=` is a member of the set of predicate symbols $\mathcal{P}$, then the
     interpretation of  the equality symbol `=` has to be *canonical*, i.e. we must have 
     $$ =^\mathcal{J} \;=\; \bigl\{ \langle u, u \rangle \mid u \in \mathcal{U} \bigr\}. $$
     
A $\Sigma$-structure $\mathcal{S} = \langle \mathcal{U}, \mathcal{J} \rangle$ is represented as an object of the following **Java** *record* that has three components:
* `universe` is the universe $\mathcal{U}$,
* `functions` is a `Map` that maps every function symbol $f$ to a `Map` $f^{\mathcal{J}}$. In turn,
  $f^{\mathcal{J}}$ maps tuples of numbers to numbers.
* `predicates` is a `Map` that maps every $n$-ary predicate symbol $p$ to a `RecursiveSet` $p^{\mathcal{J}}$. In turn,
  $p^{\mathcal{J}}$ is a set of tuples of length $n$.

In [ ]:
record Structure(RecursiveSet<Integer>                          universe,
                 Map<String, Map<List<Integer>, Integer>>       functions,
                 Map<String, RecursiveSet<List<Integer>>>       predicates) {}

Assume a signature 
$$\Sigma = \langle \mathcal{V}, \mathcal{F}, \mathcal{P}, \texttt{arity} \rangle$$
is given.  Furthermore,  $\mathcal{S} = \langle \mathcal{U}, \mathcal{J} \rangle$ is a  $\Sigma$-structure.
Then a function of the form 
$$ \mathcal{I}: \mathcal{V} \rightarrow \mathcal{U} $$
is an *$\mathcal{S}$ variable assignment*.

This function is represented as a `Map<String, Integer>` mapping variable names to numbers.

We define the universe `U` as the set $\{0, 1\}$:

In [ ]:
var U = set(0, 1);

We define interpretations for the constant `e`  and the binary function symbol `*`.

The interpretation of the constant `e` maps the empty tuple to the number 0.

In [ ]:
Map<List<Integer>, Integer> NeutralElement = Map.of(List.of(), 0);
NeutralElement

In [ ]:
Map<List<Integer>, Integer> Product = new LinkedHashMap<>();
Product.put(List.of(0, 0), 0);
Product.put(List.of(0, 1), 1);
Product.put(List.of(1, 0), 1);
Product.put(List.of(1, 1), 0);
Product

The interpretation of `=` is the set $\bigl\{ \langle x, x \rangle \mid x \in \mathcal{U} \bigr\}$.

In [ ]:
var Identity = U.map(x -> List.of(x, x));
Identity

We proceed to define the interpretation $\mathcal{J}$.  This interpretation is split in the interpretation of the function symbols and the interpretation of the predicate symbols.

In [ ]:
Map<String, Map<List<Integer>, Integer>> functions = new LinkedHashMap<>();
functions.put("e", NeutralElement);
functions.put("*", Product);
functions

In [ ]:
Map<String, RecursiveSet<List<Integer>>> predicates = new LinkedHashMap<>();
predicates.put("=", Identity);
predicates

In [ ]:
var S = new Structure(U, functions, predicates);
S

We define a variable assignment $\mathcal{I}$.  The concrete definition given here is arbitrary, because the axioms for group theory contain no free variables.

In [ ]:
Map<String, Integer> I = new TreeMap<>(Map.of("X", 0, "Y", 1, "Z", 0));
I

## Evaluation Functions

If $\Sigma = \langle \mathcal{V}, \mathcal{F}, \mathcal{P}, \texttt{arity} \rangle$ is a signature
$\mathcal{S} = \langle\mathcal{U},\mathcal{J}\rangle$ is a  $\Sigma$-structure and $\mathcal{I}$ is an  $\mathcal{S}$
variable assignment, then for every term $t$ the *value $\mathcal{S}(\mathcal{I}, t)$*
is defined by induction on $t$:
 + If $x \in \mathcal{V}$ we define:
   $$\mathcal{S}(\mathcal{I}, x) := \mathcal{I}(x). $$
 + If $c \in \mathcal{F}$ and $\texttt{arity}(c) = 0$, then 
   $$\mathcal{S}(\mathcal{I}, c) := c^\mathcal{J}. $$
 + If $t$ is a term of the form $f(t_1,\cdots,t_n)$, then we define 
   $$\mathcal{S}\bigl(\mathcal{I}, f(t_1,\cdots,t_n)\bigr) := 
                           f^\mathcal{J}\bigl( \mathcal{S}(\mathcal{I}, t_1), \cdots, \mathcal{S}(\mathcal{I}, t_n) \bigr). $$

The function `evalTerm`$(t, \mathcal{S}, \mathcal{I})$ takes
 - a term $t$,
 - a structure $\mathcal{S}$, and
 - a variable assignment $\mathcal{I}$,
and computes the value $\mathcal{S}(\mathcal{I}, t)$, i.e. it evaluates the term $t$ in the structure $\mathcal{S}$ with the variable assignment $\mathcal{I}$. 

In [ ]:
int evalTerm(Term t, Structure S, Map<String, Integer> I) {
    return switch (t) {
        case Var(var x) -> I.get(x);
        case App(var f, var args) -> {
            List<Integer> argValues = args.stream().map(arg -> evalTerm(arg, S, I)).toList();
            var fJ = S.functions().get(f);
            yield fJ.get(argValues);
        }
        default -> throw new IllegalArgumentException("unknown term: " + t);
    };
}

The function `modify`$(\mathcal{I}, x, c)$ takes
 - a variable assignment $\mathcal{I}$,
 - a variable $x$, and 
 - a value $c$, 
as inputs and computes the modified variable assignement $\mathcal{I}[x/c]$.  Here,
$\mathcal{I}[x/c]$ denotes the variable assignment
that maps the variable  $x$ to the value $c$ and that otherwise agrees with $\mathcal{I}$
$$\mathcal{I}[x/c](y) := \left\{
    \begin{array}{ll}
    c               & \mbox{if}\; y = x;  \\
    \mathcal{I}(y)  & \mbox{otherwise}.          \\
    \end{array}
    \right.
$$

In [ ]:
Map<String, Integer> modify(Map<String, Integer> I, String x, int c) {
    var Is = new TreeMap<>(I);
    Is.put(x, c);
    return Is;
}

The function `evalFormula`$(F, \mathcal{S}, \mathcal{I})$ takes
 - a formula $F$,
 - a structure $\mathcal{S}$, and
 - a variable assignment $\mathcal{I}$
as inputs and computes the value $\mathcal{S}(\mathcal{I}, F)$, i.e. it evaluates the formula $F$ in the structure $\mathcal{S}$ with the variable assignment $\mathcal{I}$.
 
Let $\mathcal{S} = \langle \mathcal{U}, \mathcal{J} \rangle$ be a $\Sigma$-structure and $\mathcal{I}$ a $\mathcal{S}$-variable 
assignment.  Then the value $\mathcal{S}(\mathcal{I},F)$ is defined by induction:
  * $\mathcal{S}(\mathcal{I},\top) := \mathtt{true}$ and $\mathcal{S}(\mathcal{I},\bot) := \mathtt{false}$.
  * If $p$ is a propositional variable, then 
    $$\mathcal{S}(\mathcal{I}, p) := p^\mathcal{J}.$$
  * $\mathcal{S}\bigl(\mathcal{I}, p(t_1,\cdots,t_n)\bigr) := 
          \Bigl(\bigl\langle \mathcal{S}(\mathcal{I}, t_1), \cdots, \mathcal{S}(\mathcal{I}, t_n) \bigr\rangle \in p^\mathcal{J}\Bigr)$.
  * $\mathcal{S}(\mathcal{I}, \neg F) := \require{enclose} \enclose{circle}{\neg}\bigl(\mathcal{S}(\mathcal{I}, F)\bigr)$.
  * $\mathcal{S}(\mathcal{I}, F \wedge G) := \require{enclose} \enclose{circle}\wedge\bigl(\mathcal{S}(\mathcal{I}, F), \mathcal{S}(\mathcal{I}, G)\bigr)$.
  * $\mathcal{S}(\mathcal{I}, F \vee G) := \require{enclose} \enclose{circle}\vee\bigl(\mathcal{S}(\mathcal{I}, F), \mathcal{S}(\mathcal{I}, G)\bigr)$.
  * $\mathcal{S}(\mathcal{I}, F \rightarrow G) := \require{enclose} \enclose{circle}\rightarrow\bigl(\mathcal{S}(\mathcal{I}, F), \mathcal{S}(\mathcal{I}, G)\bigr)$.
  * $\mathcal{S}(\mathcal{I}, F \leftrightarrow G) := \require{enclose} \enclose{circle}\leftrightarrow\bigl(\mathcal{S}(\mathcal{I}, F), \mathcal{S}(\mathcal{I}, G)\bigr)$.
  * $\mathcal{S}\bigl(\mathcal{I}, \forall x\colon F\bigr) := \left\{
      \begin{array}{ll}
         \mathtt{true}  & \mbox{if}\; \mathcal{S}(\mathcal{I}[x/c], F) = \mathtt{true}\quad \mbox{holds for all}\; c\in \mathcal{U}; \\
         \mathtt{false} & \mbox{otherwise}.
      \end{array}
      \right.$
   * $\mathcal{S}\bigl(\mathcal{I}, \exists x \colon F\bigr) := \left\{
      \begin{array}{ll}
         \mathtt{true}  & \mbox{if}\; \mathcal{S}(\mathcal{I}[x/c], F) = \mathtt{true}\quad \mbox{holds for some}\; c\in \mathcal{U}; \\
         \mathtt{false} & \mbox{otherwise}.
      \end{array}
      \right.$

In [ ]:
boolean evalFormula(Formula F, Structure S, Map<String, Integer> I) {
    return switch (F) {
        case Verum()               -> true;
        case Falsum()              -> false;
        case Not(var G)            -> !evalFormula(G, S, I);
        case And(var G, var H)     -> evalFormula(G, S, I) && evalFormula(H, S, I);
        case Or(var G, var H)      -> evalFormula(G, S, I) || evalFormula(H, S, I);
        case Implies(var G, var H) -> !evalFormula(G, S, I) || evalFormula(H, S, I);
        case Equiv(var G, var H)   -> evalFormula(G, S, I) == evalFormula(H, S, I);
        case ForAll(var x, var G)  -> S.universe().every(c -> evalFormula(G, S, modify(I, x, c)));
        case Exists(var x, var G)  -> S.universe().some (c -> evalFormula(G, S, modify(I, x, c)));
        case Atom(var p, var args) -> {
            List<Integer> argValues = args.stream().map(arg -> evalTerm(arg, S, I)).toList();
            var pJ = S.predicates().get(p);
            yield pJ.contains(argValues);
        }
        default -> throw new IllegalArgumentException("unknown formula: " + F);
    };
}

## Checking whether $\mathcal{S}$ is a Group

In [ ]:
S

In [ ]:
System.out.println("evalFormula(G1) = " + evalFormula(F1, S, I));
System.out.println("evalFormula(G2) = " + evalFormula(F2, S, I));
System.out.println("evalFormula(G3) = " + evalFormula(F3, S, I));
System.out.println("evalFormula(G4) = " + evalFormula(F4, S, I));

## Another Example

Let's show that the formula $\forall X: \exists Y: p(X,Y) \rightarrow \exists Y: \forall X: p(X,Y)$ is not universally valid.

In [ ]:
var FInvalid = parse("∀X: ∃Y: p(X, Y) → ∃Y: ∀X: p(X, Y)");
FInvalid

In [ ]:
var U2 = set(0, 1);

RecursiveSet<List<Integer>> pJ = set(List.of(0, 1), List.of(1, 0));
Map<String, RecursiveSet<List<Integer>>> predicates2 = new LinkedHashMap<>();
predicates2.put("p", pJ);

var S2 = new Structure(U2, new LinkedHashMap<>(), predicates2);
S2

In [ ]:
Map<String, Integer> I2 = new TreeMap<>(Map.of("X", 0, "Y", 0));
I2

In [ ]:
System.out.println("Result: " + evalFormula(FInvalid, S2, I2));